# Write a program for pre-processing of a text document such as stop word removal, stemming.
**Computer Laboratory II (417526) – Information Retrieval – Practical 1**

Install once (Linux terminal):
```
pip install nltk scikit-learn pandas
```

In [ ]:
import os, re, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
warnings.filterwarnings("ignore")

DATA_DIR = "data"
def get_file(name, url):
    # Return path to data/<name>; download it first if it is not there yet.
    os.makedirs(DATA_DIR, exist_ok=True)
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print("Downloading", url)
        urllib.request.urlretrieve(url, path)
    return path

In [ ]:
import nltk
# NLTK resources (needs internet once). The notebook still works without them using the fallbacks below.
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print("could not download", pkg, "->", e)

## Practical 1 – Pre-processing of a text document
**Pipeline:** sentence segmentation → tokenization → lower-casing → stop-word removal → stemming → lemmatization → POS tagging. The output of each stage is the input of the next.

* **Stop words** are very frequent words (*is, the, are, of, in, and…*) that carry little meaning, so they are removed.
* **Stemming** chops a word to its stem using rules (`learning → learn`, `studies → studi`).
* **Lemmatization** uses a dictionary and the part of speech to return a real word (`studies → study`, `better → good`).

In [ ]:
from nltk.tokenize import sent_tokenize, RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer

# Offline copy of NLTK's English stop-word list (used only if the NLTK data cannot be downloaded)
NLTK_STOP_FALLBACK = set('''i me my myself we our ours ourselves you you're you've you'll you'd your yours yourself
yourselves he him his himself she she's her hers herself it it's its itself they them their theirs themselves what which
who whom this that that'll these those am is are was were be been being have has had having do does did doing a an the
and but if or because as until while of at by for with about against between into through during before after above below
to from up down in out on off over under again further then once here there when where why how all any both each few more
most other some such no nor not only own same so than too very s t can will just don don't should should've now d ll m o re
ve y ain aren aren't couldn couldn't didn didn't doesn doesn't hadn hadn't hasn hasn't haven haven't isn isn't ma mightn
mightn't mustn mustn't needn needn't shan shan't shouldn shouldn't wasn wasn't weren weren't won won't wouldn wouldn't'''.split())

text = '''Natural Language Processing (NLP) is a field of Artificial Intelligence. It helps computers read, process
and understand human language! Dr. Smith studies how machines are learning to translate languages better.
Search engines apply stemming for retrieving documents that match the queries of users.'''

# --- the stop-word list (NLTK, with an offline fallback) ---
try:
    STOP = set(stopwords.words("english"))
except LookupError:
    STOP = set(NLTK_STOP_FALLBACK)
    print("NLTK stopwords not available -> using the built-in copy of NLTK's English stop-word list")

# 1. Sentence segmentation
try:
    sentences = sent_tokenize(text)
except LookupError:
    sentences = re.split(r'(?<!\bDr\.)(?<!\bMr\.)(?<!\bMrs\.)(?<!\bMs\.)(?<!\bProf\.)(?<=[.!?])\s+(?=[A-Z])',
                             text.replace("\n", " "))
print("1. SENTENCES:")
for s in sentences: print("   -", s)

# 2. Tokenization (words only, punctuation dropped)
tokenizer = RegexpTokenizer(r"[A-Za-z]+")
tokens = tokenizer.tokenize(text)
print("\n2. TOKENS:", tokens)

# 3. Lower-casing
tokens = [t.lower() for t in tokens]
print("\n3. LOWER-CASE:", tokens)

# 4. Stop-word removal
filtered = [t for t in tokens if t not in STOP]
print("\n4. WITHOUT STOP WORDS:", filtered)
print("   removed", len(tokens) - len(filtered), "of", len(tokens), "tokens")

In [ ]:
# 5. Stemming
porter = PorterStemmer()
snowball = SnowballStemmer("english")
stems = pd.DataFrame({"word": filtered,
                      "Porter": [porter.stem(w) for w in filtered],
                      "Snowball": [snowball.stem(w) for w in filtered]})
print("5. STEMMING"); print(stems.drop_duplicates().head(20).to_string(index=False))

# 6. Lemmatization (needs WordNet)
try:
    lem = WordNetLemmatizer()
    lem.lemmatize("cats")                       # triggers LookupError if wordnet is missing
    lemmas = [(w, lem.lemmatize(w), lem.lemmatize(w, pos="v")) for w in filtered]
    print("\n6. LEMMATIZATION (as noun, as verb)")
    print(pd.DataFrame(lemmas, columns=["word", "lemma(noun)", "lemma(verb)"]).drop_duplicates().head(20).to_string(index=False))
    print("\n   stem vs lemma:", porter.stem("studies"), "vs", lem.lemmatize("studies"),
          "|", porter.stem("better"), "vs", lem.lemmatize("better", pos="a"))
except LookupError:
    print("\n6. WordNet data not available - run  nltk.download('wordnet')  once with internet access")

# 7. POS tagging (needs the tagger model)
try:
    print("\n7. POS TAGS:", nltk.pos_tag(filtered[:12]))
except LookupError:
    print("\n7. POS tagger data not available - run  nltk.download('averaged_perceptron_tagger_eng')")

In [ ]:
# Reusable function and a file based example (as in the lab: read a text file and clean it)
def preprocess(text, stem=True, remove_stop=True):
    toks = [t.lower() for t in RegexpTokenizer(r"[A-Za-z]+").tokenize(text)]
    if remove_stop:
        toks = [t for t in toks if t not in STOP]
    if stem:
        toks = [porter.stem(t) for t in toks]
    return toks

with open("sample.txt", "w") as f:
    f.write(text)
with open("sample.txt", encoding="utf8") as f:
    print(preprocess(f.read()))